
## Architecture
**Stage 1 - Label Generation (Unsupervised)**  
K-Means clustering and rule-based thresholds are applied independently to four engineered timestamp features. Subscribers where both methods agree are assigned high-confidence labels (Version A). Rule-based labels alone form a broader label set (Version B).

**Stage 2 - Classifier Training (Supervised)**  
Two Random Forest classifiers are trained on timestamp features against Version A and Version B labels respectively.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

In [ ]:
# Read dataset
df = pd.read_csv('subscriber_activity_aggregated.csv')
print(df.info())
display(df.describe())
df.head()

K-Means uses Euclidean distance, so features with larger numeric ranges dominate distance calculations over features with smaller ranges. To handle this, StandardScaler is chosen as the normalization method instead of log transform. Log transform compresses the long tail to achieve a bell-shaped distribution, but K-Means deals with distance and it standardizes each feature to mean 0 and standard deviation 1. However, with extreme outliers present, StandardScaler inflates the standard deviation significantly, compressing normal values into a narrow z-score range. The four engineered features are visualized first to assess distribution shape before making the final scaling decision.

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))

features = ['clks_b4_opn_prop', 'opn_to_first_clk_gap_avg', 
            'min_clk_sec_gap_avg', 'click_counts_avg']

for ax, col in zip(axs.flat, features):    
    ax.hist(df[col], bins=50)
    ax.set_title(col)
    ax.set_xlabel('value')
    ax.set_ylabel('count')

plt.tight_layout()
plt.show()

All four features are heavily right-skewed with high concentration near zero and long tails toward extreme values. `min_clk_sec_gap_avg` shows artificial bimodality from NaN replacement, where the second peak represents zero-click and single-click subscribers filled at the 99th percentile.

Given that, RobustScaler is chosen to normalize feature scales before K-Means clustering. Unlike StandardScaler, RobustScaler uses median and IQR instead of mean and standard deviation, making it highly resilient to the outliers as seen in all four features.

In [ ]:
# Initialize and fit the scaler
scaler = RobustScaler()
scaled_fe = scaler.fit_transform(df[['clks_b4_opn_prop', 'min_clk_sec_gap_avg', 'click_counts_avg']]) # excluding open to click column, discussed after K-Means
scaled_fe

The negative values is mathematically expected because it scales features by removing the median and dividing by the IQR.

### K-Means Modeling

Two methods are used to select k before fitting the final model. The elbow method plots inertia, the sum of squared distances between each point and its cluster centroid, against k. The point where inertia stops dropping steeply indicates diminishing returns from adding more clusters.

In [ ]:
# Elbow method
# Inertia
inertia_list = []

# k range
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_int runs K-Means 10 times with different random initializations and keep the best result
    kmeans.fit(scaled_fe)
    inertia_list.append(kmeans.inertia_)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_list, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

The silhouette score measures how similar each point is to its own cluster relative to other clusters. It ranges from -1 to +1, higher values indicate better-defined cluster separation. The k with the peak silhouette score is selected.

In [ ]:
# Silhouette score
sil_list = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_fe)
    score = silhouette_score(scaled_fe, kmeans.labels_)
    sil_list.append(score)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, sil_list, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. k')
plt.show()

The elbow method and silhouette score were run across k=2 through k=10 to select the number of clusters before fitting the final model.

The elbow plot shows a steep inertia drop from k=2 to k=3, after which the curve flattens and additional clusters are negligible. The silhouette score peaks at k=2 and declines as k increases, indicating that two clusters produce the strongest separation relative to cohesion.

Both metrics agree on k=2, which aligns with the goal of the analysis: the two expected behavioral groups are security gateway bots and human subscribers. Beyond k=2, additional clusters partition existing groups rather than capture new behavioral patterns.

In [ ]:
# Fit final model
kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans2.fit(scaled_fe)

# Assign cluster labels
df['kmeans_label'] = kmeans2.labels_
print(df.info())
print(df['kmeans_label'].value_counts())

In [ ]:
cluster_profile = df.groupby('kmeans_label')[['clks_b4_opn_prop', 'opn_to_first_clk_gap_avg', 
                      'min_clk_sec_gap_avg', 'click_counts_avg']].mean()
display(cluster_profile.round(4))

In [ ]:
# Profile each cluster to identify which cluster represents bot behavior
df['kmeans_bot'] = (df['kmeans_label'] == 1).astype(int)
print(df['kmeans_bot'].value_counts())

`opn_to_first_clk_gap_avg` was excluded from clustering. Initial runs showed K-Means separating on the cap boundary of this feature rather than on behavioral differences between subscribers. The feature is retained for rule-based labeling though.

##### K-Means might be undersegmenting
K-Means separated on `clks_b4_opn_prop`. The dominant split in feature space is between subscribers with nonzero click-before-open behavior and everyone else. Subscribers with high click counts but zero `clks_b4_opn_prop` sit in the human cluster. K-Means has no way to flag them as suspicious based on click volume alone when their timing behavior looks human.

Domain-based classification was considered and rejected as NSTEM's subscriber base is predominantly the education sector, making domain type an unreliable bot classification for this specific list. Labels are instead created from rule-based thresholds grounded in known bot behavioral characteristics: clicks before opens, near-zero open-click and click latency, and abnormally high click counts per session.

##### Rule-based labeling
Rule-based labeling applies hard thresholds independent of K-Means, including on `click_counts_avg` and `opn_to_first_clk_gap_avg`. Thresholds are set conservatively to capture high-confidence bot behavior only, given what is physically or behaviorally possible for a human subscriber. As long as a subscriber satisifes any one of the four conditions, they will be flagged as bot behavior.

Four conditional thresholds were applied:
- `clks_b4_opn_prop` > 0 - any session with a click recorded before an email open is impossible for a human subscriber
- `opn_to_first_clk_gap_avg` < 2 - clicking within 2 seconds of opening an email means no time to read before acting
- `min_clk_sec_gap_avg` < 5 - consecutive clicks under 5 seconds apart are not realistic with human behavior
- `click_counts_avg` > 4 - averaging more than 4 clicks per session suggests automated link scanning rather than genuine engagement

In [ ]:
# Defining the four thresholds
rule_clks_b4_opn = df['clks_b4_opn_prop'] > 0
rule_opn_to_clk = df['opn_to_first_clk_gap_avg'] < 2
rule_min_clk_gap = df['min_clk_sec_gap_avg'] < 5
rule_click_counts = df['click_counts_avg'] > 4

print(f'clks_b4_opn:  {rule_clks_b4_opn.sum()}')
print(f'opn_to_clk:   {rule_opn_to_clk.sum()}')
print(f'min_clk_gap:  {rule_min_clk_gap.sum()}')
print(f'click_counts: {rule_click_counts.sum()}')


In [ ]:
# Applying thresholds
df['rule_bot'] = (
    rule_clks_b4_opn |
    rule_opn_to_clk |
    rule_min_clk_gap |
    rule_click_counts
).astype(int)

df['rule_bot'].value_counts()

There will be two versions of Random Forest Classifier. Version A is the combined labels of `kmeans_bots` and `rule_bot` labels. When both methods agreed, a subscriber is bot behavior with high confidence. When they disagree, the subscriber is uncertain and excluded from the classifier training. Version B is solely rule-based labels. 

In [ ]:
# Version A, Combined Label
# Only subscribers flagged by both K-Means and rule-based
combined_mask = (df['kmeans_bot'] == 1) & (df['rule_bot'] == 1)
human_mask = (df['kmeans_bot'] == 0) & (df['rule_bot'] == 0)

df['ver_a_bot'] = np.nan
df.loc[combined_mask, 'ver_a_bot'] = 1
df.loc[human_mask, 'ver_a_bot'] = 0

print('Version A label counts:')
print(df['ver_a_bot'].value_counts(dropna=False))

In [ ]:
# Version B, rule-based labels only
# All subscribers flagged by rule-based thresholds
df['ver_b_bot'] = df['rule_bot'].copy()

print('Version B label counts:')
print(df['ver_b_bot'].value_counts(dropna=False))

In [ ]:
# Version B training set
# Defining features for the model input
features = ['clks_b4_opn_prop', 'opn_to_first_clk_gap_avg', 
            'min_clk_sec_gap_avg', 'click_counts_avg']

X = df[features] # the input
y = df['ver_b_bot'] # the labels we are predicting

# Train/test split 
# stratify preserves class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,   # hold out 20% of the data for evaluation
    random_state=42, # seeds the randomness so results are reproducible
    stratify=y       # ensures bot/human ratio is preserved in both train and test sets
)

# Fit Random Forest
rf_label_b = RandomForestClassifier(
    n_estimators=100,        # setting the model to run 100 decision trees
    class_weight='balanced', # upweights bot class to compensate for imbalance
    random_state=42          # seeds bootstrap sampling and feature selection for reproducibility
)

# Train the model
rf_label_b.fit(X_train, y_train)



In [ ]:
# Checking score on training data
y_train_pred = rf_label_b.predict(X_train)
print(f'F1: {f1_score(y_train, y_train_pred)}')

In [ ]:
# Evaluate on the 20% test data
y_pred = rf_label_b.predict(X_test)

print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1:        {f1_score(y_test, y_pred):.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.title('Random Forest - Version B')
plt.tight_layout()
plt.show()

The dataset was split 80/20 using stratified sampling to preserve the bot/human class ratio across both sets. The held-out 20%  was used exclusively for evaluation.

**Results on held-out test set:**
- *Trained F1: 1.0000*
- Precision: 1.0000     - perfect score at not flagging human as bots
- Recall: 0.9780        - very good at flagging actual bots
- F1: 0.9889            - gap of 0.011, no overfitting
- Zero false positives  - no humans incorrectly flagged as bot

Version B labels were generated by the rule-based layer alone with no subscribers excluded. The classifier was trained on the same four features used to create the labels. Near-perfect scores reflect internal consistency rather than performance on unseen data. The model rediscovers the threshold logic used to generate its own training labels.

In [ ]:
# Version A training set
df_A = df.dropna(subset=['ver_a_bot']).copy()

X_A = df_A[features]
y_A = df_A['ver_a_bot']

X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X_A, y_A, test_size=0.2, random_state=42, stratify=y_A
)

rf_label_a = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
rf_label_a.fit(X_train_A, y_train_A)

In [ ]:
# Evaluate on the 20% test data
y_pred_A = rf_label_a.predict(X_test_A)

print(f'Precision: {precision_score(y_test_A, y_pred_A):.4f}')
print(f'Recall:    {recall_score(y_test_A, y_pred_A):.4f}')
print(f'F1:        {f1_score(y_test_A, y_pred_A):.4f}')

# Confusion matrix
cm_A = confusion_matrix(y_test_A, y_pred_A)
disp_A = ConfusionMatrixDisplay(confusion_matrix=cm_A, display_labels=[0, 1])
disp_A.plot()
plt.title('Random Forest - Version A')
plt.tight_layout()
plt.show()

**Results on held-out test set:**
- *Trained F1: 1.0000* 
- Precision: 1.0000
- Recall: 1.0000
- F1: 1.0000
- Zero false positives, zero false negatives

Version A labels were generated by combining K-Means and rule-based results. Only subscribers where both methods agreed were assigned a label — disagreements were excluded as uncertain. This is the more conservative label set. The same label-feature circularity limitation from Version B applies here, but the bot class contains only the most unambiguous behavioral signatures. Perfect scores are expected and confirm the labeling logic is internally consistent, not that the model performs on unseen data.

In [ ]:
# Save model with version a labels 
joblib.dump(rf_label_a, 'models/bot_classifier_v1_combined.pkl')

# Save model with version b labels 
joblib.dump(rf_label_b, 'models/bot_classifier_v1_rulebased.pkl')